In [0]:
use catalog ecommerce_dev;
use schema gold;

#FACT TABLE CREATION#

In [0]:
create table if not exists fact_payment
(
   order_id string,
   total_payment_value double,
   payment_type string
);

-- payment_clean , payment_enhnaced

In [0]:
create table if not exists fact_order_items(
  order_item_id int,
  order_id string,
  product_id string,
  seller_id string,
  total_item_value double
);

-- order_item_enhanced

In [0]:
create table if not exists fact_order(
  order_id string,
  customer_id string,
  order_purchase_timestamp timestamp,
  order_month int,
  order_year int ,
  order_status string,
  is_delivered boolean,
  is_late_delivery boolean
);

-- order_enhanced

In [0]:
create table if not exists fact_review(
  review_id string,
  product_id string, --join
  order_id  string ,
  seller_id string , --join
  review_score int ,
  review_category string
);
-- review_enhanced
--(product_id) join order_item_enhanced with order_id then map the product_id to this fact
-- (seller_id)  join order_item_enhanced with order_id then map the seller_id to this fact

#LOAD FACT TABLE# 

In [0]:
use catalog ecommerce_dev;
use schema silver;

### fact_payment load

In [0]:
delete from ecommerce_dev.gold.fact_payment;

insert into ecommerce_dev.gold.fact_payment(order_id, total_payment_value, payment_type)
select e.order_id, e.total_payment_value, c.payment_type
from ecommerce_dev.silver.payments_enhanced e
left join ecommerce_dev.silver.payments_clean c on e.order_id = c.order_id;

In [0]:
select * from ecommerce_dev.gold.fact_payment;

### fact_order_items load

In [0]:
delete from ecommerce_dev.gold.fact_order_items;

insert into ecommerce_dev.gold.fact_order_items(order_item_id,order_id , product_id ,seller_id ,
total_item_value)
select order_item_id,order_id , product_id ,seller_id ,
total_item_value from ecommerce_dev.silver.order_items_enhanced;

 


In [0]:
select * from ecommerce_dev.gold.fact_order_items;


### fact_order load

In [0]:
delete from ecommerce_dev.gold.fact_order;

insert into ecommerce_dev.gold.fact_order(order_id, customer_id, order_purchase_timestamp, order_month, order_year, order_status, is_delivered, is_late_delivery)
select order_id, customer_id, order_purchase_timestamp, order_month, order_year, order_status, is_delivered, is_late_delivery
from ecommerce_dev.silver.orders_enhanced;

In [0]:
select * from ecommerce_dev.gold.fact_order

### fact_review load

In [0]:
delete from ecommerce_dev.gold.fact_review;

insert into ecommerce_dev.gold.fact_review(review_id, product_id, order_id, seller_id, review_score , review_category)
select r.review_id , o.product_id, r.order_id, o.seller_id , r.review_score , review_category from 
ecommerce_dev.silver.reviews_enhanced r
inner join ecommerce_dev.silver.order_items_enhanced o on r.order_id = o.order_id;


In [0]:
select * from ecommerce_dev.gold.fact_review;